In [74]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
from IPython.display import display
from iapws import IAPWS97

In [75]:
data_path = Path.cwd().parent.joinpath("data")
gen_1 = os.path.join(data_path, "processed",  "gen1_clean.csv")
gen_2 = os.path.join(data_path, "processed",  "gen2_clean.csv")
gen_3 = os.path.join(data_path, "processed",  "gen3_clean.csv")


In [76]:
gen1 = pd.read_csv(gen_1)
gen2 = pd.read_csv(gen_2)
gen3 = pd.read_csv(gen_3)

# Rename sensor columns with generator prefix, drop date from gen2 and gen3
gen1 = gen1.rename(columns={c: f"gen1_{c}" for c in gen1.columns if c != 'date'})
gen2 = gen2.rename(columns={c: f"gen2_{c}" for c in gen2.columns if c != 'date'})
gen3 = gen3.rename(columns={c: f"gen3_{c}" for c in gen3.columns if c != 'date'})

# Stack sideways — one shared date column, all sensors as columns
df = pd.concat(
    [gen1, gen2.drop(columns=['date']), gen3.drop(columns=['date'])],
    axis=1  # axis=1 means sideways, not stacked
)

print(df.shape)          # should be (186, 34) — 1 date + 11*3 sensors
print(df.columns.tolist())
display(df.head(10))

(186, 34)
['date', 'gen1_load_MW', 'gen1_vent_pressure_bar', 'gen1_steam_flow_th', 'gen1_scrubber_temp_C', 'gen1_scrubber_pressure_bar', 'gen1_lh_inlet_temp_C', 'gen1_rh_inlet_temp_C', 'gen1_conductivity_uMHO', 'gen1_chest_pressure_barg', 'gen1_exhaust_pressure_bara', 'gen1_exhaust_temp_C', 'gen2_load_MW', 'gen2_vent_pressure_bar', 'gen2_steam_flow_th', 'gen2_scrubber_temp_C', 'gen2_scrubber_pressure_bar', 'gen2_lh_inlet_temp_C', 'gen2_rh_inlet_temp_C', 'gen2_conductivity_uMHO', 'gen2_chest_pressure_barg', 'gen2_exhaust_pressure_bara', 'gen2_exhaust_temp_C', 'gen3_load_MW', 'gen3_vent_pressure_bar', 'gen3_steam_flow_th', 'gen3_scrubber_temp_C', 'gen3_scrubber_pressure_bar', 'gen3_lh_inlet_temp_C', 'gen3_rh_inlet_temp_C', 'gen3_conductivity_uMHO', 'gen3_chest_pressure_barg', 'gen3_exhaust_pressure_bara', 'gen3_exhaust_temp_C']


,date,gen1_load_MW,gen1_vent_pressure_bar,gen1_steam_flow_th,gen1_scrubber_temp_C,gen1_scrubber_pressure_bar,gen1_lh_inlet_temp_C,gen1_rh_inlet_temp_C,gen1_conductivity_uMHO,gen1_chest_pressure_barg,...,gen3_vent_pressure_bar,gen3_steam_flow_th,gen3_scrubber_temp_C,gen3_scrubber_pressure_bar,gen3_lh_inlet_temp_C,gen3_rh_inlet_temp_C,gen3_conductivity_uMHO,gen3_chest_pressure_barg,gen3_exhaust_pressure_bara,gen3_exhaust_temp_C
0,2018-01-11,30.5,5.47,261.3,159.1,5.0,154.6,158.5,4.0,5.211,...,5.46,263.6,159.1,5.0,154.6,158.5,3.2,5.202,0.121,49.8
1,2018-02-11,30.4,5.51,264.3,159.1,5.0,154.7,158.6,3.0,5.248,...,5.49,217.2,159.5,5.0,155.4,159.3,6.8,4.096,0.095,44.7
2,2018-02-11,30.7,5.65,266.4,159.9,5.5,155.4,159.3,4.4,5.329,...,5.49,217.2,159.5,5.0,155.4,159.3,3.2,4.096,0.095,44.7
3,2018-03-11,25.7,5.66,237.3,150.4,5.6,156.1,159.9,3.8,4.662,...,5.69,259.2,150.3,5.6,156.1,160.0,3.8,4.657,0.140,52.4
4,2018-03-11,30.8,5.49,266.2,159.2,5.4,154.8,158.7,8.0,5.221,...,5.69,259.2,150.3,5.6,156.1,160.0,3.2,4.657,0.140,52.4
5,2018-04-11,25.7,5.67,241.3,160.2,5.6,156.0,159.0,3.4,4.600,...,5.67,241.3,160.2,5.6,156.0,159.0,3.4,4.602,0.137,52.2
6,2018-04-11,25.6,5.50,243.2,159.5,5.4,155.0,158.8,4.4,4.658,...,5.47,232.0,159.4,5.4,155.0,155.0,7.0,4.461,0.131,51.6
7,2018-05-11,30.8,5.47,260.7,158.0,5.3,154.4,158.3,3.0,5.200,...,5.47,251.9,158.9,5.3,154.6,158.4,3.4,5.053,0.111,48.0
8,2018-05-11,25.3,5.42,240.0,159.0,5.3,154.5,158.3,3.6,4.571,...,5.45,262.0,159.0,5.3,154.2,158.0,3.4,5.101,0.115,49.4
9,2018-06-11,30.6,5.53,263.5,159.5,5.4,154.7,158.7,3.0,5.235,...,5.53,265.8,158.9,5.4,154.6,158.6,3.0,5.189,0.114,48.7


In [77]:
# Save the dataset
raw_path = Path.cwd().parent / "data" / "processed"
df.to_csv(raw_path / "merged_olkaria_data.csv", index=False)

In [78]:
# Derived Physical Features
for gen in ['gen1', 'gen2', 'gen3']:
    # LH + RH are 0.92 correlated — compress into average (shared signal)
    # and asymmetry (the unique fault signal the EDA flagged)
    df[f'{gen}_inlet_temp_avg']      = (df[f'{gen}_lh_inlet_temp_C'] + df[f'{gen}_rh_inlet_temp_C']) / 2
    df[f'{gen}_inlet_temp_asymmetry'] = (df[f'{gen}_lh_inlet_temp_C'] - df[f'{gen}_rh_inlet_temp_C']).abs()

    # How much pressure is the turbine actually dropping across itself?
    df[f'{gen}_pressure_drop'] = df[f'{gen}_vent_pressure_bar'] - df[f'{gen}_exhaust_pressure_bara']

    # MW generated per tonne/hr of steam — efficiency ratio
    df[f'{gen}_steam_utilization'] = df[f'{gen}_load_MW'] / df[f'{gen}_steam_flow_th']


print(f"Shape after FE cell 1: {df.shape}")
print(df.columns.tolist())

Shape after FE cell 1: (186, 46)
['date', 'gen1_load_MW', 'gen1_vent_pressure_bar', 'gen1_steam_flow_th', 'gen1_scrubber_temp_C', 'gen1_scrubber_pressure_bar', 'gen1_lh_inlet_temp_C', 'gen1_rh_inlet_temp_C', 'gen1_conductivity_uMHO', 'gen1_chest_pressure_barg', 'gen1_exhaust_pressure_bara', 'gen1_exhaust_temp_C', 'gen2_load_MW', 'gen2_vent_pressure_bar', 'gen2_steam_flow_th', 'gen2_scrubber_temp_C', 'gen2_scrubber_pressure_bar', 'gen2_lh_inlet_temp_C', 'gen2_rh_inlet_temp_C', 'gen2_conductivity_uMHO', 'gen2_chest_pressure_barg', 'gen2_exhaust_pressure_bara', 'gen2_exhaust_temp_C', 'gen3_load_MW', 'gen3_vent_pressure_bar', 'gen3_steam_flow_th', 'gen3_scrubber_temp_C', 'gen3_scrubber_pressure_bar', 'gen3_lh_inlet_temp_C', 'gen3_rh_inlet_temp_C', 'gen3_conductivity_uMHO', 'gen3_chest_pressure_barg', 'gen3_exhaust_pressure_bara', 'gen3_exhaust_temp_C', 'gen1_inlet_temp_avg', 'gen1_inlet_temp_asymmetry', 'gen1_pressure_drop', 'gen1_steam_utilization', 'gen2_inlet_temp_avg', 'gen2_inlet_temp

In [79]:
# Temporal & Cross-Generator Features
df['date'] = pd.to_datetime(df['date'])
# --- REGIME FLAG ---
# EDA showed a permanent structural break in conductivity across all three generators
# between Nov 2018 and Jan 2019. Pre-break = stable low-conductivity steam conditions.
# Post-break = sustained contamination-prone regime with frequent critical events.
# Encoding this as a binary flag lets the model treat the two eras differently
# rather than assuming stationarity across the full 6-year period.
df['post_regime_shift'] = (df['date'] >= '2019-01-01').astype(int)

# --- TEMPORAL FEATURES ---
# Geothermal reservoirs are not fully immune to seasonal effects — rainfall patterns,
# reinjection cycles, and maintenance windows tend to cluster by month.
# EDA also showed year-on-year reservoir drift (declining capacity factors).
# Month captures intra-year seasonality; year captures long-term degradation trend.
df['month'] = df['date'].dt.month
df['year']  = df['date'].dt.year

# CROSS-GENERATOR LOAD DIVERGENCE
# All three generators draw from the same Olkaria II wellfield but their readings
# are staggered by 4 hours (Gen1 @ 08:00, Gen2 @ 12:00, Gen3 @ 04:00).
# If the reservoir were perfectly stable, all three would read similar loads on the
# same date. Divergence between them on the same date therefore directly reflects
# intra-day reservoir drift, wellhead pressure drops, enthalpy decay, and
# shifting steam-to-brine ratios that cause off-design output drops.
# This validates the off-design operating conditions the EDA quantified (~83% CF).
df['load_divergence_g1_g2'] = (df['gen1_load_MW'] - df['gen2_load_MW']).abs()
df['load_divergence_g1_g3'] = (df['gen1_load_MW'] - df['gen3_load_MW']).abs()
df['load_divergence_g2_g3'] = (df['gen2_load_MW'] - df['gen3_load_MW']).abs()

# Single severity score — the worst divergence seen across all three on a given date.
# A high value on this column means the reservoir was in an unstable state that day.
df['max_load_divergence'] = df[['load_divergence_g1_g2',
                                 'load_divergence_g1_g3',
                                 'load_divergence_g2_g3']].max(axis=1)

# --- CROSS-GENERATOR STEAM DIVERGENCE ---
# Same logic applied to steam flow. Enthalpy decay in the reservoir shows up as
# declining steam mass flow before it shows up as MW drop — so steam divergence
# is an earlier upstream signal of the same reservoir drift phenomenon.
df['steam_divergence_g1_g2'] = (df['gen1_steam_flow_th'] - df['gen2_steam_flow_th']).abs()
df['steam_divergence_g1_g3'] = (df['gen1_steam_flow_th'] - df['gen3_steam_flow_th']).abs()
df['steam_divergence_g2_g3'] = (df['gen2_steam_flow_th'] - df['gen3_steam_flow_th']).abs()

print(f"Shape after FE cell 2: {df.shape}")
display(df[['date', 'post_regime_shift', 'month', 'year',
          'load_divergence_g1_g2', 'load_divergence_g1_g3',
          'max_load_divergence']].head(8))

Shape after FE cell 2: (186, 56)


,date,post_regime_shift,month,year,load_divergence_g1_g2,load_divergence_g1_g3,max_load_divergence
0,2018-01-11,0,1,2018,0.4,0.2,0.4
1,2018-02-11,0,2,2018,5.1,4.9,5.1
2,2018-02-11,0,2,2018,0.0,5.2,5.2
3,2018-03-11,0,3,2018,0.0,0.0,0.0
4,2018-03-11,0,3,2018,5.8,5.1,5.8
5,2018-04-11,0,4,2018,0.1,0.1,0.1
6,2018-04-11,0,4,2018,1.1,0.2,1.1
7,2018-05-11,0,5,2018,0.1,0.5,0.6


In [80]:
# Rolling & Lag Features
# Sort by date first, rolling/lag features are order-dependent.
# If rows aren't in time order, you'd be leaking future data into past rows.
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

for gen in ['gen1', 'gen2', 'gen3']:

    # Lag-1: what was this variable one reading ago?
    # Captures short-term momentum — is load dropping or holding?
    df[f'{gen}_load_lag1']        = df[f'{gen}_load_MW'].shift(1)
    df[f'{gen}_steam_lag1']       = df[f'{gen}_steam_flow_th'].shift(1)
    df[f'{gen}_conductivity_lag1']= df[f'{gen}_conductivity_uMHO'].shift(1)

    # Smoothed average over last 3 readings — suppresses noise, captures trend.
    # EDA showed conductivity spikes are episodic — rolling mean separates
    # baseline from spikes better than raw values alone.
    df[f'{gen}_load_roll3']         = df[f'{gen}_load_MW'].rolling(window=3).mean()
    df[f'{gen}_conductivity_roll3'] = df[f'{gen}_conductivity_uMHO'].rolling(window=3).mean()
    df[f'{gen}_steam_roll3']        = df[f'{gen}_steam_flow_th'].rolling(window=3).mean()

    # Variability over the last 3 readings — high std = unstable operation.
    # Useful for the ANN surrogate to detect off-design volatility.
    df[f'{gen}_load_roll3_std']   = df[f'{gen}_load_MW'].rolling(window=3).std()

# First 2 rows will have NaNs in rolling/lag columns — expected, not a bug.
# We'll handle them when we prep for the model (forward-fill or drop).
print(f"Shape after FE cell 3: {df.shape}")
print(f"NaNs introduced: {df.isnull().sum().sum()}")
display(df[['date', 'gen1_load_MW', 'gen1_load_lag1', 'gen1_load_roll3', 'gen1_load_roll3_std']].head(8))

Shape after FE cell 3: (186, 77)
NaNs introduced: 33


,date,gen1_load_MW,gen1_load_lag1,gen1_load_roll3,gen1_load_roll3_std
0,2018-01-11,30.5,NaN,NaN,NaN
1,2018-01-12,28.1,30.5,NaN,NaN
2,2018-01-12,28.5,28.1,29.033333,1.285820
3,2018-02-11,30.4,28.5,29.000000,1.228821
4,2018-02-11,30.7,30.4,29.866667,1.193035
5,2018-02-12,27.7,30.7,29.600000,1.652271
6,2018-02-12,28.1,27.7,28.833333,1.628906
7,2018-03-11,25.7,28.1,27.166667,1.285820


In [81]:
#Steam Table Thermodynamic Properties (FIXED)


# Dead state reference (25°C, 1 atm)
T0_K = 25 + 273.15
P0_MPa = 0.101325
dead_state = IAPWS97(T=T0_K, P=P0_MPa)
h0 = dead_state.h
s0 = dead_state.s
print(f"Dead state: h0={h0:.2f} kJ/kg, s0={s0:.4f} kJ/kg·K")

results = []

for _, row in df.iterrows():
    for gen in ['gen1', 'gen2', 'gen3']:

        # --- STATE 2: Scrubber inlet ---
        # Use temperature only with x=0 (saturated liquid = brine)
        # and x=1 (saturated steam) — avoids T/P inconsistency errors
        T2_K = row[f'{gen}_scrubber_temp_C'] + 273.15
        sat2_liq = IAPWS97(T=T2_K, x=0)   # brine leaving separator
        sat2_stm = IAPWS97(T=T2_K, x=1)   # steam going to turbine
        h2_f = sat2_liq.h
        s2_f = sat2_liq.s
        h2_g = sat2_stm.h

        # --- STATE 3: Turbine inlet ---
        # Use temperature only, assume saturated steam (x=1) at inlet
        # Geothermal steam after scrubber is saturated — not superheated
        T3_K = ((row[f'{gen}_lh_inlet_temp_C'] + row[f'{gen}_rh_inlet_temp_C']) / 2) + 273.15
        state3 = IAPWS97(T=T3_K, x=1)     # saturated steam at turbine inlet
        h3 = state3.h
        s3 = state3.s

        # --- STATE 5: Turbine exhaust ---
        # Exhaust is wet steam (two-phase). Use temperature only with x=1
        # to get saturated vapor enthalpy, then account for quality
        # Typical geothermal turbine exhaust quality ~0.88-0.92
        T5_K = row[f'{gen}_exhaust_temp_C'] + 273.15
        sat5_liq = IAPWS97(T=T5_K, x=0)
        sat5_stm = IAPWS97(T=T5_K, x=1)
        h5_f = sat5_liq.h
        h5_fg = sat5_stm.h - sat5_liq.h
        s5_f = sat5_liq.s
        s5_fg = sat5_stm.s - sat5_liq.s

        # Estimate exhaust quality from isentropic expansion assumption
        # s5 = s3 for isentropic, then x5 = (s3 - s5_f) / s5_fg
        x5 = min(1.0, max(0.8, (s3 - s5_f) / s5_fg))  # clamp between 0.8-1.0
        h5 = h5_f + x5 * h5_fg
        s5 = s5_f + x5 * s5_fg

        # --- TURBINE WORK ---
        m_dot = row[f'{gen}_steam_flow_th'] / 3.6  # T/H → kg/s
        W_isentropic_kW = m_dot * (h3 - h5)
        W_isentropic_MW = W_isentropic_kW / 1000

        # Turbine isentropic efficiency = actual / isentropic
        measured_MW = row[f'{gen}_load_MW']
        turbine_efficiency = measured_MW / W_isentropic_MW if W_isentropic_MW > 0 else None

        # --- EXERGY AT TURBINE INLET ---
        ex3 = (h3 - h0) - T0_K * (s3 - s0)
        Ex_in_kW = m_dot * ex3
        Ex_in_MW = Ex_in_kW / 1000

        # --- EXERGY EFFICIENCY ---
        exergy_efficiency = measured_MW / Ex_in_MW if Ex_in_MW > 0 else None

        # --- BRINE EXERGY (second flash potential) ---
        ex2_f = (h2_f - h0) - T0_K * (s2_f - s0)
        # Brine mass flow ≈ total steam flow × brine fraction
        # At ~158°C scrubber inlet, typical steam quality ~0.2 → brine = 80%
        m_dot_brine = m_dot * 0.8
        Ex_brine_MW = (m_dot_brine * ex2_f) / 1000

        # Clip turbine efficiency to physically valid range
        thermo_df['turbine_isentropic_eff'] = thermo_df['turbine_isentropic_eff'].clip(upper=1.0)

        results.append({
            'date': row['date'],
            'generator': gen,
            'h3_kJ_kg': round(h3, 2),
            'h5_kJ_kg': round(h5, 2),
            'h2_brine_kJ_kg': round(h2_f, 2),
            'x5_exhaust_quality': round(x5, 4),
            's3_kJ_kgK': round(s3, 4),
            'W_isentropic_MW': round(W_isentropic_MW, 3),
            'turbine_isentropic_eff': round(turbine_efficiency, 4) if turbine_efficiency else None,
            'exergy_in_MW': round(Ex_in_MW, 3),
            'exergy_efficiency': round(exergy_efficiency, 4) if exergy_efficiency else None,
            'brine_exergy_MW': round(Ex_brine_MW, 3),
        })

thermo_df = pd.DataFrame(results)
print(f"Shape: {thermo_df.shape}")
display(thermo_df.describe().round(3))
display(thermo_df.head(6))

Dead state: h0=104.93 kJ/kg, s0=0.3672 kJ/kg·K
Shape: (558, 12)


,date,h3_kJ_kg,h5_kJ_kg,h2_brine_kJ_kg,x5_exhaust_quality,s3_kJ_kgK,W_isentropic_MW,turbine_isentropic_eff,exergy_in_MW,exergy_efficiency,brine_exergy_MW
count,558,558.000,558.000,558.000,558.000,558.000,558.000,558.000,558.000,558.000,558.000
mean,2019-01-21 00:54:11.612903,2752.859,2172.281,668.736,0.824,6.784,39.879,0.742,50.463,0.586,5.427
min,2018-01-11 00:00:00,2747.000,2130.430,633.550,0.813,6.757,3.991,0.559,5.184,0.440,0.554
25%,2018-11-14 00:00:00,2752.720,2165.210,668.190,0.822,6.780,39.268,0.698,49.761,0.556,5.348
50%,2018-12-20 12:00:00,2753.060,2169.400,669.710,0.824,6.783,39.762,0.710,50.374,0.563,5.402
75%,2019-03-03 00:00:00,2753.460,2177.212,671.230,0.826,6.786,40.728,0.723,51.164,0.573,5.510
max,2023-11-28 00:00:00,2756.440,2219.620,678.620,0.837,6.829,47.027,6.138,59.977,4.726,6.450
std,NaN,1.324,12.873,5.800,0.004,0.010,2.213,0.241,2.633,0.184,0.300


,date,generator,h3_kJ_kg,h5_kJ_kg,h2_brine_kJ_kg,x5_exhaust_quality,s3_kJ_kgK,W_isentropic_MW,turbine_isentropic_eff,exergy_in_MW,exergy_efficiency,brine_exergy_MW
0,2018-01-11,gen1,2753.57,2173.14,671.66,0.8244,6.7790,42.130,0.7240,53.492,0.5702,5.787
1,2018-01-11,gen2,2753.80,2167.71,672.10,0.8228,6.7772,42.345,0.7108,53.301,0.5647,5.769
2,2018-01-11,gen3,2753.57,2171.32,671.66,0.8239,6.7790,42.634,0.7107,53.963,0.5615,5.838
3,2018-01-12,gen1,2753.06,2161.62,667.32,0.8215,6.7829,40.087,0.7010,49.837,0.5638,5.331
4,2018-01-12,gen2,2753.06,2166.50,669.06,0.8228,6.7829,39.756,0.7144,49.837,0.5699,5.360
5,2018-01-12,gen3,2752.83,2164.62,668.19,0.8223,6.7847,40.194,0.7041,50.194,0.5638,5.389


In [82]:
# Merge thermodynamic features back into main df
# thermo_df is long format (one row per generator per date)
# df is wide format (one row per date, all generators as columns)
# Strategy: pivot thermo_df wide then merge on date

thermo_wide = thermo_df.pivot_table(
    index='date',
    columns='generator',
    values=['h3_kJ_kg', 'h5_kJ_kg', 'h2_brine_kJ_kg',
            'x5_exhaust_quality', 'W_isentropic_MW',
            'turbine_isentropic_eff', 'exergy_in_MW',
            'exergy_efficiency', 'brine_exergy_MW']
)

# Flatten column names: (feature, gen) → gen_feature
thermo_wide.columns = [f'{gen}_{feat}' for feat, gen in thermo_wide.columns]
thermo_wide = thermo_wide.reset_index()

df = df.merge(thermo_wide, on='date', how='left')

print(f"Shape after merging thermodynamics: {df.shape}")
print([c for c in df.columns if 'exergy' in c or 'isentropic' in c])

Shape after merging thermodynamics: (186, 104)
['gen1_W_isentropic_MW', 'gen2_W_isentropic_MW', 'gen3_W_isentropic_MW', 'gen1_brine_exergy_MW', 'gen2_brine_exergy_MW', 'gen3_brine_exergy_MW', 'gen1_exergy_efficiency', 'gen2_exergy_efficiency', 'gen3_exergy_efficiency', 'gen1_exergy_in_MW', 'gen2_exergy_in_MW', 'gen3_exergy_in_MW', 'gen1_turbine_isentropic_eff', 'gen2_turbine_isentropic_eff', 'gen3_turbine_isentropic_eff']


In [83]:
df.head(10)

,date,gen1_load_MW,gen1_vent_pressure_bar,gen1_steam_flow_th,gen1_scrubber_temp_C,gen1_scrubber_pressure_bar,gen1_lh_inlet_temp_C,gen1_rh_inlet_temp_C,gen1_conductivity_uMHO,gen1_chest_pressure_barg,...,gen3_h3_kJ_kg,gen1_h5_kJ_kg,gen2_h5_kJ_kg,gen3_h5_kJ_kg,gen1_turbine_isentropic_eff,gen2_turbine_isentropic_eff,gen3_turbine_isentropic_eff,gen1_x5_exhaust_quality,gen2_x5_exhaust_quality,gen3_x5_exhaust_quality
0,2018-01-11,30.5,5.47,261.3,159.1,5.0,154.6,158.5,4.0,5.211,...,2753.570,2173.140,2167.710,2171.320,0.72400,0.71080,0.71070,0.82440,0.82280,0.82390
1,2018-01-12,28.1,5.40,244.0,158.1,5.3,154.0,158.2,3.2,5.130,...,2752.890,2164.480,2166.780,2164.785,0.70965,0.70840,0.71035,0.82230,0.82290,0.82235
2,2018-01-12,28.5,5.39,244.0,158.6,5.3,153.8,157.8,3.4,5.118,...,2752.890,2164.480,2166.780,2164.785,0.70965,0.70840,0.71035,0.82230,0.82290,0.82235
3,2018-02-11,30.4,5.51,264.3,159.1,5.0,154.7,158.6,3.0,5.248,...,2754.480,2172.180,2158.210,2137.930,0.71230,0.69450,0.68550,0.82390,0.82005,0.81460
4,2018-02-11,30.7,5.65,266.4,159.9,5.5,155.4,159.3,4.4,5.329,...,2754.480,2172.180,2158.210,2137.930,0.71230,0.69450,0.68550,0.82390,0.82005,0.81460
5,2018-02-12,27.7,5.35,241.6,158.3,5.3,154.0,158.0,2.8,5.135,...,2752.835,2166.850,2165.935,2164.620,0.70195,0.70605,0.69850,0.82290,0.82270,0.82230
6,2018-02-12,28.1,5.38,246.7,158.5,5.3,154.0,157.9,3.2,5.130,...,2752.835,2166.850,2165.935,2164.620,0.70195,0.70605,0.69850,0.82290,0.82270,0.82230
7,2018-03-11,25.7,5.66,237.3,150.4,5.6,156.1,159.9,3.8,4.662,...,2755.270,2178.685,2183.550,2182.830,0.70010,0.66990,0.62360,0.82550,0.82675,0.82630
8,2018-03-11,30.8,5.49,266.2,159.2,5.4,154.8,158.7,8.0,5.221,...,2755.270,2178.685,2183.550,2182.830,0.70010,0.66990,0.62360,0.82550,0.82675,0.82630
9,2018-03-12,27.9,5.37,247.7,158.4,5.2,153.8,157.7,3.6,5.106,...,2752.775,2168.655,2170.440,2167.200,0.69900,0.70185,0.69680,0.82345,0.82390,0.82310


In [84]:
cols_to_drop = [c for c in df.columns if c.endswith('_lh_inlet_temp_C') or c.endswith('_rh_inlet_temp_C')]
df = df.drop(columns=cols_to_drop)

In [85]:
# Save final engineered dataset
output_path = data_path / "processed" / "olkaria_features_engineered.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {df.shape} → {output_path}")

Saved: (186, 98) → /home/mlops-localhost/PycharmProjects/double-flash-geothermal/data/processed/olkaria_features_engineered.csv
